# SKU Percentage Finder

## Importing Library

In [354]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np

## Import Necessary File

In [355]:
JNP_File=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Import files\JNP\CompleteJNP_Data.csv"
PDM_File=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Import files\PDM\CompletePDM_Data.csv"
AutocareAttrFile=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Autocare\Reference\df_full.csv"


## Dataframe Creation

In [356]:
df_s=pd.read_csv(PDM_File)

In [357]:
df_s

,BrandName,PartType,Attribute,AttributeUomLabel,PartNumber,AttributeValue
0,BBDW_ASC,Engine Water Pump,Fan Clutch Included,NaN,WP-1000,No
1,BBDW_ASC,Engine Water Pump,Housing Material,NaN,WP-1000,Cast Iron
2,BBDW_ASC,Engine Water Pump,Hub Height,MM,WP-1000,113.03
3,BBDW_ASC,Engine Water Pump,Hub Hole Quantity,NaN,WP-1000,4
4,BBDW_ASC,Engine Water Pump,Hub Hole Thread Size,NaN,WP-1000,M6X1.00
...,...,...,...,...,...,...
6664168,T083_MEX Autolite,Windshield Wiper Blade,SMALL ENGINE,NaN,UR-8,N
6664169,T083_MEX Autolite,Windshield Wiper Blade,SNOWMOBILE,NaN,UR-8,N
6664170,T083_MEX Autolite,Windshield Wiper Blade,TRANSMISSION SPECIFIC,NaN,UR-8,N
6664171,T083_MEX Autolite,Windshield Wiper Blade,Design,NaN,UR-8,For heavy rains


### Custom Cells (for PDM)

In [358]:
print(df_s.columns)
df_s['AttributeUomLabel']=df_s['AttributeUomLabel'].str.lower()
df_s["Attribute_Full"] = np.where(df_s["AttributeUomLabel"].notna(), df_s['Attribute'] + " ("+df_s['AttributeUomLabel']+")", df_s["Attribute"])
df_s=df_s.astype(str)

df_s.rename(columns={
'PartType': 'PartName',
'AttributeValue': 'Value',
},inplace=True)



Index(['BrandName', 'PartType', 'Attribute', 'AttributeUomLabel', 'PartNumber',
       'AttributeValue'],
      dtype='object')


In [359]:
df_s['Key']=df_s['PartNumber']+df_s['PartName']+df_s['Attribute_Full']

In [360]:
df_s

,BrandName,PartName,Attribute,AttributeUomLabel,PartNumber,Value,Attribute_Full,Key
0,BBDW_ASC,Engine Water Pump,Fan Clutch Included,nan,WP-1000,No,Fan Clutch Included,WP-1000Engine Water PumpFan Clutch Included
1,BBDW_ASC,Engine Water Pump,Housing Material,nan,WP-1000,Cast Iron,Housing Material,WP-1000Engine Water PumpHousing Material
2,BBDW_ASC,Engine Water Pump,Hub Height,mm,WP-1000,113.03,Hub Height (mm),WP-1000Engine Water PumpHub Height (mm)
3,BBDW_ASC,Engine Water Pump,Hub Hole Quantity,nan,WP-1000,4,Hub Hole Quantity,WP-1000Engine Water PumpHub Hole Quantity
4,BBDW_ASC,Engine Water Pump,Hub Hole Thread Size,nan,WP-1000,M6X1.00,Hub Hole Thread Size,WP-1000Engine Water PumpHub Hole Thread Size
...,...,...,...,...,...,...,...,...
6664168,T083_MEX Autolite,Windshield Wiper Blade,SMALL ENGINE,nan,UR-8,N,SMALL ENGINE,UR-8Windshield Wiper BladeSMALL ENGINE
6664169,T083_MEX Autolite,Windshield Wiper Blade,SNOWMOBILE,nan,UR-8,N,SNOWMOBILE,UR-8Windshield Wiper BladeSNOWMOBILE
6664170,T083_MEX Autolite,Windshield Wiper Blade,TRANSMISSION SPECIFIC,nan,UR-8,N,TRANSMISSION SPECIFIC,UR-8Windshield Wiper BladeTRANSMISSION SPECIFIC
6664171,T083_MEX Autolite,Windshield Wiper Blade,Design,nan,UR-8,For heavy rains,Design,UR-8Windshield Wiper BladeDesign


In [361]:
df_Brands=df_s[['PartNumber','BrandName']].drop_duplicates(ignore_index=True)
df_Brands

,PartNumber,BrandName
0,WP-1000,BBDW_ASC
1,WP-1088,BBDW_ASC
2,WP-1090,BBDW_ASC
3,WP-1091,BBDW_ASC
4,WP-1103,BBDW_ASC
...,...,...
192560,UR-131,T083_MEX Autolite
192561,UR-14,T083_MEX Autolite
192562,UR-15,T083_MEX Autolite
192563,UR-16,T083_MEX Autolite


In [362]:
df_BrandMap=pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Autocare\Active_Part_List.xlsx",sheet_name="BrandMap")
df_BrandMap=df_BrandMap[["BrandName","Brand"]]
df_BrandMap.dropna(inplace=True)
df_BrandMap

,BrandName,Brand
0,BPI,BPI
1,Cardone,Cardone New
2,Centric,Centric
3,IBI,IBI
4,StopTech,Stop Tech
5,CNRD_CARDONE New,Cardone New
9,CPCX_CARDONE Reman,Cardone Reman
12,BBDW_ASC,Airtex
13,HYJT_Carter Water Pumps,Carter
15,BBZN_Strong Arm,Strong Arm


In [363]:
df_PNS=df_s[['PartNumber','PartName','BrandName']].drop_duplicates(ignore_index=True)

## AutocareList

In [364]:
df_AC=pd.read_csv(AutocareAttrFile)

In [365]:
df_AC

,PAPTID,PartTerminologyID,PAID,MetaID,PartTerminologyName,PartsDescriptionId,RevDate_x,PAName,PADescr,MetaUomCodeAssignmentID,...,UOMCode,UOMDescription,UOMLabel,MeasurementGroupId,CodeMasterID,CategoryID,SubCategoryID,PositionID,RevDate_y,CategoryName
0,55603.0,1020.0,14.0,88.0,Car Cover,29761.0,2021-09-15,Length,Describes Measurement,34876.0,...,LM,Meter,m,1.0,1.0,1.0,457.0,1.0,2003-02-07,Accessories
1,55604.0,1020.0,24.0,88.0,Car Cover,29761.0,2021-09-15,Width,Describes Measurement,34879.0,...,LM,Meter,m,1.0,1.0,1.0,457.0,1.0,2003-02-07,Accessories
2,55603.0,1020.0,14.0,88.0,Car Cover,29761.0,2021-09-15,Length,Describes Measurement,34874.0,...,IN,Inch,in,1.0,1.0,1.0,457.0,1.0,2003-02-07,Accessories
3,55604.0,1020.0,24.0,88.0,Car Cover,29761.0,2021-09-15,Width,Describes Measurement,34877.0,...,IN,Inch,in,1.0,1.0,1.0,457.0,1.0,2003-02-07,Accessories
4,55603.0,1020.0,14.0,88.0,Car Cover,29761.0,2021-09-15,Length,Describes Measurement,34875.0,...,FT,Foot,ft,1.0,1.0,1.0,457.0,1.0,2003-02-07,Accessories
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
860323,NaN,NaN,11283.0,NaN,NaN,NaN,NaN,Oil Supply Line Included,Describes whether or not the oil supply line i...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
860324,NaN,NaN,11284.0,NaN,NaN,NaN,NaN,Oil Drain Line Included,Describes whether or not the oil drain line is...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
860325,NaN,NaN,11285.0,NaN,NaN,NaN,NaN,Coolant Supply Line Included,Describes whether or not a coolant supply line...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
860326,NaN,NaN,11286.0,NaN,NaN,NaN,NaN,Coolant Drain Line Included,Describes whether or not a coolant drain line ...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [366]:
df_AC=df_AC[[ 'PartTerminologyName','PAName', 'UOMLabel']]

In [367]:
df_AC.rename(columns={'PartTerminologyName': 'PartName','PAName': 'Attribute'},inplace=True)

In [368]:
#df_AC.dropna(subset=['Attribute']).reset_index(drop=True)

In [369]:
df_AC["Attribute_Full"] = np.where(df_AC["UOMLabel"].notna(), df_AC['Attribute'] + " ("+df_AC['UOMLabel']+")", df_AC["Attribute"])

In [370]:
df_AC=df_AC.drop_duplicates(ignore_index=True)

In [371]:
df_AC['Attribute_Type']="Autocare_Attribute"

In [372]:
df_AC

,PartName,Attribute,UOMLabel,Attribute_Full,Attribute_Type
0,Car Cover,Length,m,Length (m),Autocare_Attribute
1,Car Cover,Width,m,Width (m),Autocare_Attribute
2,Car Cover,Length,in,Length (in),Autocare_Attribute
3,Car Cover,Width,in,Width (in),Autocare_Attribute
4,Car Cover,Length,ft,Length (ft),Autocare_Attribute
...,...,...,...,...,...
210461,NaN,Oil Supply Line Included,NaN,Oil Supply Line Included,Autocare_Attribute
210462,NaN,Oil Drain Line Included,NaN,Oil Drain Line Included,Autocare_Attribute
210463,NaN,Coolant Supply Line Included,NaN,Coolant Supply Line Included,Autocare_Attribute
210464,NaN,Coolant Drain Line Included,NaN,Coolant Drain Line Included,Autocare_Attribute


## Merging

In [373]:
df_merged=df_PNS.merge(df_AC,how='left')
df_merged.shape

(5954360, 7)

In [374]:
df_merged

,PartNumber,PartName,BrandName,Attribute,UOMLabel,Attribute_Full,Attribute_Type
0,WP-1000,Engine Water Pump,BBDW_ASC,Hub Height,mm,Hub Height (mm),Autocare_Attribute
1,WP-1000,Engine Water Pump,BBDW_ASC,Outside Pulley Diameter,mm,Outside Pulley Diameter (mm),Autocare_Attribute
2,WP-1000,Engine Water Pump,BBDW_ASC,Hub Hole Thread Diameter,mm,Hub Hole Thread Diameter (mm),Autocare_Attribute
3,WP-1000,Engine Water Pump,BBDW_ASC,Hub Height,in,Hub Height (in),Autocare_Attribute
4,WP-1000,Engine Water Pump,BBDW_ASC,Outside Pulley Diameter,in,Outside Pulley Diameter (in),Autocare_Attribute
...,...,...,...,...,...,...,...
5954355,UR-8,Windshield Wiper Blade,T083_MEX Autolite,Adapter Type,NaN,Adapter Type,Autocare_Attribute
5954356,UR-8,Windshield Wiper Blade,T083_MEX Autolite,Wiper Blade Connection Type,NaN,Wiper Blade Connection Type,Autocare_Attribute
5954357,UR-8,Windshield Wiper Blade,T083_MEX Autolite,Frame Material,NaN,Frame Material,Autocare_Attribute
5954358,UR-8,Windshield Wiper Blade,T083_MEX Autolite,Universal Or Specific Fit,NaN,Universal Or Specific Fit,Autocare_Attribute


In [375]:
df_merged['Key']=df_merged['PartNumber']+df_merged['PartName']+df_merged['Attribute_Full']

In [376]:
df_merged

,PartNumber,PartName,BrandName,Attribute,UOMLabel,Attribute_Full,Attribute_Type,Key
0,WP-1000,Engine Water Pump,BBDW_ASC,Hub Height,mm,Hub Height (mm),Autocare_Attribute,WP-1000Engine Water PumpHub Height (mm)
1,WP-1000,Engine Water Pump,BBDW_ASC,Outside Pulley Diameter,mm,Outside Pulley Diameter (mm),Autocare_Attribute,WP-1000Engine Water PumpOutside Pulley Diamete...
2,WP-1000,Engine Water Pump,BBDW_ASC,Hub Hole Thread Diameter,mm,Hub Hole Thread Diameter (mm),Autocare_Attribute,WP-1000Engine Water PumpHub Hole Thread Diamet...
3,WP-1000,Engine Water Pump,BBDW_ASC,Hub Height,in,Hub Height (in),Autocare_Attribute,WP-1000Engine Water PumpHub Height (in)
4,WP-1000,Engine Water Pump,BBDW_ASC,Outside Pulley Diameter,in,Outside Pulley Diameter (in),Autocare_Attribute,WP-1000Engine Water PumpOutside Pulley Diamete...
...,...,...,...,...,...,...,...,...
5954355,UR-8,Windshield Wiper Blade,T083_MEX Autolite,Adapter Type,NaN,Adapter Type,Autocare_Attribute,UR-8Windshield Wiper BladeAdapter Type
5954356,UR-8,Windshield Wiper Blade,T083_MEX Autolite,Wiper Blade Connection Type,NaN,Wiper Blade Connection Type,Autocare_Attribute,UR-8Windshield Wiper BladeWiper Blade Connecti...
5954357,UR-8,Windshield Wiper Blade,T083_MEX Autolite,Frame Material,NaN,Frame Material,Autocare_Attribute,UR-8Windshield Wiper BladeFrame Material
5954358,UR-8,Windshield Wiper Blade,T083_MEX Autolite,Universal Or Specific Fit,NaN,Universal Or Specific Fit,Autocare_Attribute,UR-8Windshield Wiper BladeUniversal Or Specifi...


In [377]:
df_Final=df_merged.merge(df_s,how="outer")
df_Final.shape

(9798620, 10)

In [378]:
df_Final

,PartNumber,PartName,BrandName,Attribute,UOMLabel,Attribute_Full,Attribute_Type,Key,AttributeUomLabel,Value
0,04L 965 567,Engine Water Pump,BBDW_ASC,Casting Number,NaN,Casting Number,Autocare_Attribute,04L 965 567Engine Water PumpCasting Number,NaN,NaN
1,04L 965 567,Engine Water Pump,BBDW_ASC,Fan Clutch Included,NaN,Fan Clutch Included,Autocare_Attribute,04L 965 567Engine Water PumpFan Clutch Included,NaN,NaN
2,04L 965 567,Engine Water Pump,BBDW_ASC,Grade Type,NaN,Grade Type,Autocare_Attribute,04L 965 567Engine Water PumpGrade Type,NaN,NaN
3,04L 965 567,Engine Water Pump,BBDW_ASC,Housing Material,NaN,Housing Material,Autocare_Attribute,04L 965 567Engine Water PumpHousing Material,NaN,NaN
4,04L 965 567,Engine Water Pump,BBDW_ASC,Hub Height,in,Hub Height (in),Autocare_Attribute,04L 965 567Engine Water PumpHub Height (in),NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
9798615,Z33073,Fuel Pump Tank Seal,GQWK_Import Direct Fuel Pumps,Sealing Surface Type,NaN,Sealing Surface Type,Autocare_Attribute,Z33073Fuel Pump Tank SealSealing Surface Type,nan,Flange
9798616,Z33073,Fuel Pump Tank Seal,GQWK_Import Direct Fuel Pumps,Thickness,in,Thickness (in),Autocare_Attribute,Z33073Fuel Pump Tank SealThickness (in),NaN,NaN
9798617,Z33073,Fuel Pump Tank Seal,GQWK_Import Direct Fuel Pumps,Thickness,mm,Thickness (mm),Autocare_Attribute,Z33073Fuel Pump Tank SealThickness (mm),mm,4.0
9798618,Z33073,Fuel Pump Tank Seal,GQWK_Import Direct Fuel Pumps,Width,in,Width (in),Autocare_Attribute,Z33073Fuel Pump Tank SealWidth (in),NaN,NaN


In [379]:
df_Final['Attributes']=np.where(df_Final["Attribute_Full"].notna(), df_Final["Attribute_Full"] , df_Final["Attribute"])
df_Final=df_Final[['PartNumber', 'BrandName','PartName', 'Attributes', 'Attribute_Type', 'Value']]
df_Final['AC_AttributeCount']=np.where(df_Final["Attribute_Type"].notna(), 1 , 0)
df_Final['Attribute_Value']=np.where((df_Final["Attribute_Type"].notna())  & (df_Final["Value"].notna()), 1 , 0)
# df_Final=df_Final.dropna(subset='Autocare')

df_Final['Attribute_Type']=np.where((df_Final["Attribute_Type"].notna())  , df_Final["Attribute_Type"] , 'FBG_Attribute')

In [380]:
df_BrandMap

,BrandName,Brand
0,BPI,BPI
1,Cardone,Cardone New
2,Centric,Centric
3,IBI,IBI
4,StopTech,Stop Tech
5,CNRD_CARDONE New,Cardone New
9,CPCX_CARDONE Reman,Cardone Reman
12,BBDW_ASC,Airtex
13,HYJT_Carter Water Pumps,Carter
15,BBZN_Strong Arm,Strong Arm


In [381]:
df_Final=df_Final.merge(df_BrandMap,how="inner")
df_Final.shape

(2737343, 9)

In [382]:
df_Final

,PartNumber,BrandName,PartName,Attributes,Attribute_Type,Value,AC_AttributeCount,Attribute_Value,Brand
0,04L 965 567,BBDW_ASC,Engine Water Pump,Casting Number,Autocare_Attribute,NaN,1,0,Airtex
1,04L 965 567,BBDW_ASC,Engine Water Pump,Fan Clutch Included,Autocare_Attribute,NaN,1,0,Airtex
2,04L 965 567,BBDW_ASC,Engine Water Pump,Grade Type,Autocare_Attribute,NaN,1,0,Airtex
3,04L 965 567,BBDW_ASC,Engine Water Pump,Housing Material,Autocare_Attribute,NaN,1,0,Airtex
4,04L 965 567,BBDW_ASC,Engine Water Pump,Hub Height (in),Autocare_Attribute,NaN,1,0,Airtex
...,...,...,...,...,...,...,...,...,...
2737338,XST458DP,HXLZ_Autolite Xtreme Start,Spark Plug,Skirt Length (mm),Autocare_Attribute,NaN,1,0,Autolite
2737339,XST458DP,HXLZ_Autolite Xtreme Start,Spark Plug,Thread Diameter (in),Autocare_Attribute,NaN,1,0,Autolite
2737340,XST458DP,HXLZ_Autolite Xtreme Start,Spark Plug,Thread Diameter (mm),Autocare_Attribute,14,1,1,Autolite
2737341,XST458DP,HXLZ_Autolite Xtreme Start,Spark Plug,Washer Included,Autocare_Attribute,NaN,1,0,Autolite


In [383]:
df_Final=df_Final[['PartNumber', 'PartName', 'Attributes', 'Attribute_Type',
       'Value', 'AC_AttributeCount', 'Attribute_Value', 'Brand']] #,'BrandName'

In [384]:
df_Final

,PartNumber,PartName,Attributes,Attribute_Type,Value,AC_AttributeCount,Attribute_Value,Brand
0,04L 965 567,Engine Water Pump,Casting Number,Autocare_Attribute,NaN,1,0,Airtex
1,04L 965 567,Engine Water Pump,Fan Clutch Included,Autocare_Attribute,NaN,1,0,Airtex
2,04L 965 567,Engine Water Pump,Grade Type,Autocare_Attribute,NaN,1,0,Airtex
3,04L 965 567,Engine Water Pump,Housing Material,Autocare_Attribute,NaN,1,0,Airtex
4,04L 965 567,Engine Water Pump,Hub Height (in),Autocare_Attribute,NaN,1,0,Airtex
...,...,...,...,...,...,...,...,...
2737338,XST458DP,Spark Plug,Skirt Length (mm),Autocare_Attribute,NaN,1,0,Autolite
2737339,XST458DP,Spark Plug,Thread Diameter (in),Autocare_Attribute,NaN,1,0,Autolite
2737340,XST458DP,Spark Plug,Thread Diameter (mm),Autocare_Attribute,14,1,1,Autolite
2737341,XST458DP,Spark Plug,Washer Included,Autocare_Attribute,NaN,1,0,Autolite


In [385]:
df_Final=df_Final.drop_duplicates()
df_Final.reset_index(inplace=True,drop=True)
df_Final.shape

(2635390, 8)

In [386]:
df_Final_all=df_Final 

In [387]:
df_Final_all['Key']=df_Final_all["PartNumber"] + " " + df_Final_all["PartName"] + " " + df_Final_all["Attributes"]

In [388]:
df_Final_all

,PartNumber,PartName,Attributes,Attribute_Type,Value,AC_AttributeCount,Attribute_Value,Brand,Key
0,04L 965 567,Engine Water Pump,Casting Number,Autocare_Attribute,NaN,1,0,Airtex,04L 965 567 Engine Water Pump Casting Number
1,04L 965 567,Engine Water Pump,Fan Clutch Included,Autocare_Attribute,NaN,1,0,Airtex,04L 965 567 Engine Water Pump Fan Clutch Included
2,04L 965 567,Engine Water Pump,Grade Type,Autocare_Attribute,NaN,1,0,Airtex,04L 965 567 Engine Water Pump Grade Type
3,04L 965 567,Engine Water Pump,Housing Material,Autocare_Attribute,NaN,1,0,Airtex,04L 965 567 Engine Water Pump Housing Material
4,04L 965 567,Engine Water Pump,Hub Height (in),Autocare_Attribute,NaN,1,0,Airtex,04L 965 567 Engine Water Pump Hub Height (in)
...,...,...,...,...,...,...,...,...,...
2635385,XST458DP,Spark Plug,Pre-Gap Size (mm),Autocare_Attribute,0.8,1,1,Autolite,XST458DP Spark Plug Pre-Gap Size (mm)
2635386,XST458DP,Spark Plug,Reach (mm),Autocare_Attribute,9.5,1,1,Autolite,XST458DP Spark Plug Reach (mm)
2635387,XST458DP,Spark Plug,Resistance,FBG_Attribute,1,0,0,Autolite,XST458DP Spark Plug Resistance
2635388,XST458DP,Spark Plug,Skirt,Autocare_Attribute,NaN,1,0,Autolite,XST458DP Spark Plug Skirt


In [389]:
df_Final_all_filtered = df_Final_all.sort_values(by='Value', na_position='last').drop_duplicates(subset=['Key'], keep='first')

In [390]:
df_Final_all_filtered.shape

(2592360, 9)

In [398]:
df_Final_all_filtered

,PartNumber,PartName,Attributes,Attribute_Type,Value,AC_AttributeCount,Attribute_Value,Brand,Key
527058,19-B3405,Disc Brake Caliper,Casting Number,Autocare_Attribute,"""ATE 264""",1,1,Cardone Reman,19-B3405 Disc Brake Caliper Casting Number
526979,19-B3404,Disc Brake Caliper,Casting Number,Autocare_Attribute,"""ATE 264""",1,1,Cardone Reman,19-B3404 Disc Brake Caliper Casting Number
447915,19-B1645A,Disc Brake Caliper,Casting Number,Autocare_Attribute,"""L""",1,1,Cardone Reman,19-B1645A Disc Brake Caliper Casting Number
433811,19-B1445,Disc Brake Caliper,Casting Number,Autocare_Attribute,"""SEE PRINT""",1,1,Cardone Reman,19-B1445 Disc Brake Caliper Casting Number
433732,19-B1444,Disc Brake Caliper,Casting Number,Autocare_Attribute,"""SEE PRINT""",1,1,Cardone Reman,19-B1444 Disc Brake Caliper Casting Number
...,...,...,...,...,...,...,...,...,...
2635364,XST458DP,Spark Plug,Resistance (Ohms),Autocare_Attribute,NaN,1,0,Autolite,XST458DP Spark Plug Resistance (Ohms)
2635368,XST458DP,Spark Plug,Skirt Length (in),Autocare_Attribute,NaN,1,0,Autolite,XST458DP Spark Plug Skirt Length (in)
2635369,XST458DP,Spark Plug,Skirt Length (mm),Autocare_Attribute,NaN,1,0,Autolite,XST458DP Spark Plug Skirt Length (mm)
2635370,XST458DP,Spark Plug,Thread Diameter (in),Autocare_Attribute,NaN,1,0,Autolite,XST458DP Spark Plug Thread Diameter (in)


In [391]:
chunk_size=1000000
# Create a list of DataFrames by splitting the original DataFrame
df_chunks = [df_Final_all_filtered.iloc[i:i + chunk_size] for i in range(0, len(df_Final_all_filtered), chunk_size)]

In [392]:
df_Final=df_Final.groupby(['Brand','PartNumber','PartName'], as_index=False).agg(
    Total_Autocare_Attributes=('AC_AttributeCount','sum'),
    Autocare_Attributes_Currently_Available=('Attribute_Value','sum'),
    Total_Available_Attributes=('Attribute_Type','count')
)

In [393]:
df_Final['Filled%']=df_Final['Autocare_Attributes_Currently_Available']/df_Final['Total_Autocare_Attributes']
df_Final['FBG_Attributes']=df_Final['Total_Available_Attributes']-df_Final['Autocare_Attributes_Currently_Available']

In [394]:
df_Final

,Brand,PartNumber,PartName,Total_Autocare_Attributes,Autocare_Attributes_Currently_Available,Total_Available_Attributes,Filled%,FBG_Attributes
0,AVM Industries,4000,Trunk Lid Lift Support,40,13,40,0.325000,27
1,AVM Industries,4002,Trunk Lid Lift Support,40,14,40,0.350000,26
2,AVM Industries,4003,Hood Lift Support,41,14,41,0.341463,27
3,AVM Industries,4004,Trunk Lid Lift Support,40,13,40,0.325000,27
4,AVM Industries,4006,Hood Lift Support,41,14,41,0.341463,27
...,...,...,...,...,...,...,...,...
50846,Strong Arm,SA3001,Multi-Purpose Lift Support Stud,24,1,27,0.041667,26
50847,Strong Arm,SA3002,Multi-Purpose Lift Support Stud,24,3,26,0.125000,23
50848,Strong Arm,SA3100,Multi-Purpose Lift Support Bracket,4,3,4,0.750000,1
50849,Strong Arm,SA3101,Multi-Purpose Lift Support Bracket,4,3,4,0.750000,1


## Cleaning and Exporting.

In [395]:
Source="PDM"
Brand="Repair"
Outputfile=fr'C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Import files\{Source}\{Brand}_SKU_AttrPerc.xlsx'

In [397]:
with pd.ExcelWriter(Outputfile) as writer:  # doctest: +SKIP
    for i, chunk in enumerate(df_chunks):
        sheet_name = f"Chunk_{i+1}"  # Naming each sheet dynamically
        chunk.to_excel(writer, sheet_name=sheet_name, index=False)
    df_Final.to_excel(writer,index=False, sheet_name='Fillinginfo')